# KWS Negative sentences
Notebook for selecting and visualizing negative sentences (that contain no keyword).

In [1]:
from tira_kws.constants import (
    CAPSTONE_KEYWORDS,
    CAPSTONE_NEGATIVE_RECORDS,
    RECORD_LIST_CSV,
)

import pandas as pd
from unidecode import unidecode

/home/markjos/projects/kws_tira/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
kw_df = pd.read_csv(CAPSTONE_KEYWORDS)
kw_df

,keyword,keyword_id,gloss
0,àpɾí,0.0,boy
1,ðɔ̀mɔ̀cɔ̀,1.0,man
2,lɛ́ŋgɛ́n,2.0,mother
3,ŋɔ̀ɽíŋgɔ́,3.0,donkey
4,mùðù,4.0,leopard
5,ùɾnɔ̀,5.0,grandfather/grandchild
6,ðàŋàl,6.0,sheep
7,kə̀və̀lɛ̀ðɔ́,7.0,CLg-pull.PFV-VENT
8,kàŋú,8.0,CLg-see.PFV-VENT
9,íŋgá_nɔ̀nà,9.0,1sg-CLg-AUX_see.IPFV-IT


In [3]:
record_df = pd.read_csv(RECORD_LIST_CSV)
record_df.head()

,record_idx,recording_id,start,duration,channel,text,language,speaker,gloss,lemmata,translation,fst_normalized,unidecode_normalized
0,0,HH01082021,217.011,2.541,0,àprí jɜ̀dí ðáŋàlà,tic,Himidan,"boy[NOM,SG] touch[CLJ,VENT,PFV] sheep[ACC,SG,l...",àpɾí ɜd ðàŋàl,boy held sheep,àpɾí jɜ̀dí ðáŋàlà,apri jedi dangala
1,1,HH01082021,221.371,3.652,0,àprí jɜ̀dí ðáŋàlà,tic,Himidan,"boy[NOM,SG] touch[CLJ,VENT,PFV] sheep[ACC,SG,l...",àpɾí ɜd ðàŋàl,boy held sheep,àpɾí jɜ̀dí ðáŋàlà,apri jedi dangala
2,2,HH01082021,283.401,3.127,0,àprí jə̀və̀lɛ̀ðɔ́ ðáŋàlà,tic,Himidan,"boy[NOM,SG] pull[CLJ,VENT,PFV] sheep[ACC,SG,le...",àpɾí vəlɛð ðàŋàl,boy pulled sheep (towards),àpɾí jə̀və̀lɛ̀ðɔ́ ðáŋàlà,apri j@v@ledo dangala
3,3,HH01082021,288.835,2.745,0,àprí jə̀və̀lɛ̀ðɔ́ ðáŋàlà,tic,Himidan,"boy[NOM,SG] pull[CLJ,VENT,PFV] sheep[ACC,SG,le...",àpɾí vəlɛð ðàŋàl,boy pulled sheep (towards),àpɾí jə̀və̀lɛ̀ðɔ́ ðáŋàlà,apri j@v@ledo dangala
4,4,HH01082021,304.737,3.025,0,àprí jə̀və̀lɛ̀ðɔ́ ðáŋàlà,tic,Himidan,"boy[NOM,SG] pull[CLJ,VENT,PFV] sheep[ACC,SG,le...",àpɾí vəlɛð ðàŋàl,boy pulled sheep (towards),àpɾí jə̀və̀lɛ̀ðɔ́ ðáŋàlà,apri j@v@ledo dangala


## Keyword mask
Select records that do not contain a unidecoded string for any of the 10 keywords.

In [4]:
keyword_mask = pd.Series([True]*len(record_df))

for keyword in kw_df['keyword'].unique():
    keyword_decode = unidecode(keyword)
    current_mask = ~record_df['unidecode_normalized'].str.contains(keyword_decode)
    keyword_mask &= current_mask

keyword_mask.sum()

np.int64(17824)

Now sample $n=100$ negative records and save them to CSV.

In [5]:
num_negative = 100
colmap = {
    "record_idx": "sentence_id",
    "translation": "translation",
    "text": "original_sentence",
    "fst_normalized": "textnorm_sentence",
}

negative_records = record_df[keyword_mask].sample(num_negative)
negative_records = negative_records.rename(columns=colmap)
negative_records = negative_records[colmap.values()]
negative_records.head()

,sentence_id,translation,original_sentence,textnorm_sentence
16587,16587,market is good,ìlì jɛ̀cə̀lǒ,ìlí jìcə̀lò
7223,7223,I will see the names.,íŋgánɔ̀nà ndrí,íŋgá_nɔ́nà ńdrí
14870,14870,He helped him yesterday and left,kàŋ ɔ́ɽɟɛ̀cì ánó unɛɾɛ,kàŋà ɔ́ɽɟɛ̀cì ánó únɛ́ɾɛ́
20184,20184,they will climb down (towards),lá ùrù,lá ùrú
5122,5122,They will jump tomorrow.,lábrà nd̪ɔ̀bà,lábr̀à nd̪ɔ̀bà


Load `keyword_negatives.csv` and merge data (overwrite existing rows but keep column names).

In [48]:
neg_df = pd.read_csv(CAPSTONE_NEGATIVE_RECORDS)

# neg_df = neg_df.drop(neg_df.index)
# neg_df = pd.concat([neg_df, negative_records])

neg_df.head()

,sentence_id,translation,original_sentence,textnorm_sentence,audionorm_sentence,audio_quality,extra_speech,missing_speech,mistranscription,comment
12856,12856,To feel good is good,ðícílá ðìcə̀lò,ðíbíl ðìcə̀lò,ðícílá ðìcə̀lò,3.0,False,False,False,NaN
13375,13375,To cut meat is good,ðə́ðá ɛ́ðɛ̀ ánó ðìcə̀lò,ðə́ðá ɛ́ðɛ̀ ánó ðìcə̀lò,ðə́ðá ɛ́ðɛ̀ ánó ðìcə̀lò,3.0,False,False,False,NaN
6114,6114,they opened their mouths yesterday(and left),là kíðɛ̀ ŋɔ̀ɲà ùnɛ̀rɛ̀,là kíðɛ̀ ŋɔ̀ɲà ùnɛ́ɾɛ́,là kíðɛ̀ ŋɔ̀ɲà ùnɛ́ɾɛ́,3.0,False,False,False,NaN
18219,18219,a co-wife is good,ɛ̀ɾə̀mt̪ù kìcəlò,rə̀mt̪ kìcə̀lò,ɛ̀ɾə̀mt̪ù kìcəlò,3.0,False,False,False,NaN
12417,12417,Don't help Kuku!,àt̪ ɔ́ɽɟɛ̀cɔ̀ŋnɔ̀ kúkùŋú ánó dɛ̀,át̪ú ɔ́ɽɟɛ̀cìnɔ̀ kúkùŋú ánó dɛ̀,àt̪ ɔ́ɽɟɛ̀cɔ̀ŋnɔ̀ kúkùŋú ánó dɛ̀,3.0,False,False,False,NaN


In [ ]:
# neg_df.to_csv(CAPSTONE_NEGATIVE_RECORDS)

## Revisions
A bunch of sentences with 'pull' and 'see' leaked through, let's exclude them and add back missing rows

In [49]:
# need to append rows until we have 100 negative sentences

neg_df.shape

(78, 10)

In [50]:
# filter sentences containing words that might indicate the presence of a keyword
# in the English translation

exclude_words = ["pull", "see", "look", "tickle"]

translation_mask = pd.Series([True] * len(record_df))

for word in exclude_words:
    current_mask = ~record_df["translation"].fillna("").str.lower().str.contains(word)
    translation_mask &= current_mask

# also exclude rows with no translation
translation_mask &= record_df["translation"].notna()

translation_mask.sum()

np.int64(13942)

In [51]:
# filter out rows already assigned as negative sentences

remaining_mask = ~record_df["record_idx"].isin(neg_df["sentence_id"])
remaining_mask.sum()

np.int64(20402)

In [43]:
valid_record_mask = keyword_mask & translation_mask & remaining_mask
valid_record_mask.sum()

np.int64(12192)

Sample valid sentences to reach 100 total negative sentences.

In [47]:
num_negative = 100
num_remaining = num_negative - len(neg_df)

additional_negatives = record_df[valid_record_mask].sample(num_remaining)
additional_negatives = additional_negatives.rename(columns=colmap)
additional_negatives = additional_negatives[colmap.values()]
additional_negatives.head()

,sentence_id,translation,original_sentence,textnorm_sentence
16460,16460,the goats are good,jèɽò jìclò,èɽò jìcə̀lò
18250,18250,The jujube fruit is good.,lùɽbí lìcə̀lò,lùɽbí lìcə̀lò
7929,7929,This hyena is good.\n,ðàlgàráì ðìcə̀lò,ðàlgàrâj ðìcə̀lò
7554,7554,The wives are good.,lájlɛ́n lìcə̀lò,lajlɛ́n lìcə̀lò
17142,17142,walk fast!,árlát̪ɔ́ t̪ə̀màn,áɾlát̪ɔ́ t̪ə̀mànì


Reinsert to dataframe and save.

In [46]:
total_neg_df = pd.concat([neg_df, additional_negatives], ignore_index=True)
print(total_neg_df.shape)
total_neg_df.head()

(99, 10)


,sentence_id,translation,original_sentence,textnorm_sentence,audionorm_sentence,audio_quality,extra_speech,missing_speech,mistranscription,comment
0,12856,To feel good is good,ðícílá ðìcə̀lò,ðíbíl ðìcə̀lò,ðícílá ðìcə̀lò,3.0,False,False,False,NaN
1,13375,To cut meat is good,ðə́ðá ɛ́ðɛ̀ ánó ðìcə̀lò,ðə́ðá ɛ́ðɛ̀ ánó ðìcə̀lò,ðə́ðá ɛ́ðɛ̀ ánó ðìcə̀lò,3.0,False,False,False,NaN
2,6114,they opened their mouths yesterday(and left),là kíðɛ̀ ŋɔ̀ɲà ùnɛ̀rɛ̀,là kíðɛ̀ ŋɔ̀ɲà ùnɛ́ɾɛ́,là kíðɛ̀ ŋɔ̀ɲà ùnɛ́ɾɛ́,3.0,False,False,False,NaN
3,18219,a co-wife is good,ɛ̀ɾə̀mt̪ù kìcəlò,rə̀mt̪ kìcə̀lò,ɛ̀ɾə̀mt̪ù kìcəlò,3.0,False,False,False,NaN
4,12417,Don't help Kuku!,àt̪ ɔ́ɽɟɛ̀cɔ̀ŋnɔ̀ kúkùŋú ánó dɛ̀,át̪ú ɔ́ɽɟɛ̀cìnɔ̀ kúkùŋú ánó dɛ̀,àt̪ ɔ́ɽɟɛ̀cɔ̀ŋnɔ̀ kúkùŋú ánó dɛ̀,3.0,False,False,False,NaN


In [38]:
total_neg_df.to_csv(CAPSTONE_NEGATIVE_RECORDS, index=False)